# 🏔️ Trail Difficulty & Time Predictor

Projekt z uczenia maszynowego — przewidywanie trudności i czasu przejścia szlaków górskich.

## 👤 Skład zespołu

| Imię i nazwisko | Rola |
|---|---|
| Maksymilian Kaczmarek | Autor projektu (projekt indywidualny) |

## 📋 Koncepcja projektu

Projekt składa się z **dwóch części** wykorzystujących te same dane o szlakach górskich:

### Część 1 — Klasyfikacja trudności (classification)
Przewidywanie kategorii trudności szlaku (`difficulty_rating`) na podstawie jego cech topograficznych.
- **Target:** kategoria trudności
- **4 modele klasyfikujące:** Logistic Regression, Random Forest, XGBoost, MLP
- **Metryki:** accuracy, F1-score, confusion matrix

### Część 2 — Regresja czasu przejścia (Tobler)
Ponieważ dataset nie zawiera czasu przejścia, target tworzymy **autorsko** — obliczając oczekiwany czas ze wzoru **Toblera** (fizyczny model prędkości marszu na danym nachyleniu). Następnie modele ML uczą się go odtworzyć z cech szlaku.
- **Target:** `tobler_time` (obliczony)
- **4 modele regresyjne:** Linear Regression, Random Forest, XGBoost, MLP
- **Metryki:** MAE, RMSE, R²

### Pytania badawcze
- Które cechy szlaku najmocniej decydują o jego trudności?
- Czy modele potrafią odtworzyć fizyczny model czasu marszu (Tobler) z prostych cech?
- Jak porównują się cztery różne algorytmy na tym samym problemie?

### Dane
🔗 [National Park Trails (AllTrails) — Kaggle](https://www.kaggle.com/datasets/planejane/national-park-trails)

Zbiór ~2 800 szlaków z parków narodowych USA (AllTrails, 2019).

## 🔧 Środowisko i biblioteki

In [ ]:
!pip install xgboost plotly -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Modele regresyjne
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

# Modele klasyfikujące
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Metryki
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score, confusion_matrix,
                             classification_report)

print("Biblioteki zaimportowane")

## 📥 Pobranie danych z Kaggle

In [ ]:
import kagglehub

path = kagglehub.dataset_download("planejane/national-park-trails")
print("Path to dataset files:", path)

In [ ]:
import os
print(os.listdir(path))

In [ ]:
# Wczytanie — dostosuj nazwę pliku do tego co wypisało powyżej
df = pd.read_csv(path + '/AllTrails data - nationalpark.csv')
print("Wymiary:", df.shape)
df.head()

## 🔍 Diagnostyka — co jest w danych

Sprawdzamy rzeczywiste nazwy kolumn, typy i braki, zanim zaczniemy czyszczenie.

In [ ]:
print("=== KOLUMNY ===")
print(df.columns.tolist())
print("\n=== TYPY ===")
print(df.dtypes)
print("\n=== BRAKI ===")
print(df.isnull().sum())

In [ ]:
# Rozkład targetu klasyfikacji
print(df['difficulty_rating'].value_counts())

## 🧹 Czyszczenie danych

Uwaga: w tym datasecie `length` i `elevation_gain` są zwykle w **metrach**. Sprawdzimy zakresy i usuniemy ewentualne wartości niemożliwe (zero, ujemne, ekstremalne outliery).

In [ ]:
# Podstawowe statystyki kolumn liczbowych — szukamy nonsensów
print(df[['length', 'elevation_gain']].describe())

In [ ]:
# Usuwamy szlaki o zerowej/ujemnej długości lub przewyższeniu (błędy)
df = df[(df['length'] > 0) & (df['elevation_gain'] >= 0)].copy()

# Usuwamy braki w kluczowych kolumnach
df = df.dropna(subset=['length', 'elevation_gain', 'difficulty_rating'])

print("Rekordów po czyszczeniu:", df.shape[0])

## 🛠️ Feature Engineering

Tworzymy autorskie cechy pochodne:
- `length_km` — długość w kilometrach (z metrów)
- `elevation_per_km` — przewyższenie na kilometr (proxy stromizny / średniego nachylenia)
- `avg_gradient` — średnie nachylenie (%)

In [ ]:
df['length_km'] = df['length'] / 1000
df['elevation_per_km'] = df['elevation_gain'] / df['length_km']

# Średnie nachylenie w %: przewyższenie / dystans poziomy * 100
df['avg_gradient'] = (df['elevation_gain'] / df['length']) * 100

print(df[['length_km', 'elevation_gain', 'elevation_per_km', 'avg_gradient']].describe())

## 🧮 Tworzenie targetu czasu — wzór Toblera

**Wzór Toblera** (Tobler's hiking function) szacuje prędkość marszu na podstawie nachylenia terenu:

$$ v = 6 \cdot e^{-3.5 \cdot |S + 0.05|} $$

gdzie `v` to prędkość w km/h, a `S` to nachylenie (stosunek pionu do poziomu, tangens kąta).
Maksymalna prędkość (~5 km/h) wypada przy lekkim zejściu, nie na płaskim — to zgodne z fizjologią marszu.

Czas przejścia liczymy jako: `czas = dystans / prędkość`.

In [ ]:
def tobler_time_hours(length_m, elevation_gain_m):
    """Zwraca szacowany czas przejścia szlaku w godzinach wg wzoru Toblera."""
    # nachylenie jako stosunek pionu do poziomu
    slope = elevation_gain_m / length_m
    # prędkość wg Toblera (km/h)
    speed = 6 * np.exp(-3.5 * abs(slope + 0.05))
    # dystans w km / prędkość = czas w godzinach
    length_km = length_m / 1000
    return length_km / speed

df['tobler_time'] = tobler_time_hours(df['length'], df['elevation_gain'])
print(df[['length_km', 'elevation_gain', 'avg_gradient', 'tobler_time']].head(10))
print("\nŚredni czas Toblera (h):", round(df['tobler_time'].mean(), 2))

## 🔍 Eksploracyjna Analiza Danych (EDA)

In [ ]:
# Rozkład długości szlaków
fig = px.histogram(df, x='length_km', nbins=60,
                   title='Rozkład długości szlaków',
                   labels={'length_km': 'Długość (km)'})
fig.show()

In [ ]:
# Długość vs przewyższenie, kolor wg trudności
fig = px.scatter(df, x='length_km', y='elevation_gain',
                 color='difficulty_rating',
                 title='Długość vs przewyższenie wg trudności',
                 labels={'length_km': 'Długość (km)', 'elevation_gain': 'Przewyższenie (m)'},
                 opacity=0.5)
fig.show()

In [ ]:
# Macierz korelacji cech liczbowych
corr_cols = ['length_km', 'elevation_gain', 'elevation_per_km', 'avg_gradient', 'tobler_time']
corr = df[corr_cols].corr()
fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r',
                title='Macierz korelacji cech')
fig.show()

# 🤖 CZĘŚĆ 1 — Klasyfikacja trudności

Przewidujemy `difficulty_rating` na podstawie cech szlaku. Cztery modele klasyfikujące + porównanie.

In [ ]:
# Features i target dla klasyfikacji
feature_cols = ['length_km', 'elevation_gain', 'elevation_per_km', 'avg_gradient']

X_clf = df[feature_cols]
y_clf = df['difficulty_rating']

# Skalowanie (potrzebne dla LogReg i MLP)
scaler_clf = StandardScaler()
X_clf_scaled = scaler_clf.fit_transform(X_clf)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_clf_scaled, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

print("Train:", X_tr.shape[0], "| Test:", X_te.shape[0])

In [ ]:
# XGBoost wymaga etykiet 0,1,2... więc kodujemy difficulty
le = LabelEncoder()
y_tr_enc = le.fit_transform(y_tr)
y_te_enc = le.transform(y_te)
print("Klasy:", list(le.classes_))

In [ ]:
# Trenujemy 4 klasyfikatory
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, verbosity=0),
    'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
}

clf_results = []
for name, model in classifiers.items():
    model.fit(X_tr, y_tr_enc)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te_enc, y_pred)
    f1 = f1_score(y_te_enc, y_pred, average='weighted')
    clf_results.append({'Model': name, 'Accuracy': round(acc, 4), 'F1': round(f1, 4)})

clf_df = pd.DataFrame(clf_results).sort_values('Accuracy', ascending=False)
print(clf_df)

In [ ]:
# Wykres porównania klasyfikatorów
fig = px.bar(clf_df, x='Model', y='Accuracy',
             title='Porównanie klasyfikatorów — trafność (accuracy)',
             color='Accuracy', color_continuous_scale='Greens',
             text='Accuracy')
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# Confusion matrix najlepszego modelu (Random Forest)
best_clf = classifiers['Random Forest']
y_pred_best = best_clf.predict(X_te)
cm = confusion_matrix(y_te_enc, y_pred_best)

fig = px.imshow(cm, text_auto=True, aspect='auto',
                x=le.classes_, y=le.classes_,
                color_continuous_scale='Blues',
                labels={'x': 'Predykcja', 'y': 'Rzeczywista'},
                title='Confusion Matrix — Random Forest')
fig.show()

# 🤖 CZĘŚĆ 2 — Regresja czasu (Tobler)

Przewidujemy obliczony `tobler_time` na podstawie cech szlaku. Cztery modele regresyjne + porównanie.

In [ ]:
# Features dla regresji (bez avg_gradient i elevation_per_km — by nie 'zdradzić' wzoru wprost)
reg_features = ['length_km', 'elevation_gain']

X_reg = df[reg_features]
y_reg = df['tobler_time']

scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_reg_scaled, y_reg, test_size=0.2, random_state=42)

print("Train:", Xr_tr.shape[0], "| Test:", Xr_te.shape[0])

In [ ]:
# Trenujemy 4 regresory
regressors = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'MLP': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
}

reg_results = []
for name, model in regressors.items():
    model.fit(Xr_tr, yr_tr)
    y_pred = model.predict(Xr_te)
    mae = mean_absolute_error(yr_te, y_pred)
    rmse = np.sqrt(mean_squared_error(yr_te, y_pred))
    r2 = r2_score(yr_te, y_pred)
    reg_results.append({'Model': name,
                        'MAE_min': round(mae*60, 2),
                        'RMSE_min': round(rmse*60, 2),
                        'R2': round(r2, 4)})

reg_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False)
print(reg_df)

In [ ]:
# Wykres porównania regresorów (R2)
fig = px.bar(reg_df, x='Model', y='R2',
             title='Porównanie regresorów — R² (wyższy = lepszy)',
             color='R2', color_continuous_scale='Blues',
             text='R2')
fig.update_traces(textposition='outside')
fig.show()

## 📊 Feature Importance — co decyduje o trudności

In [ ]:
# Importance z klasyfikatora Random Forest
rf_clf = classifiers['Random Forest']
imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_clf.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(imp, x='importance', y='feature', orientation='h',
             title='Ważność cech — klasyfikacja trudności',
             color='importance', color_continuous_scale='Greens')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 🏁 Wnioski końcowe

_(do uzupełnienia po uruchomieniu — omówienie wyników klasyfikacji, regresji, feature importance)_

### Część 1 — Klasyfikacja
- ...

### Część 2 — Regresja (Tobler)
- ...

### Ograniczenia
- Czas `tobler_time` to model teoretyczny, nie rzeczywiste czasy zawodników — zakłada stałe nachylenie na całej trasie.
- Dataset nie zawiera typu podłoża ani profilu wysokościowego (tylko sumaryczne przewyższenie).